# Resample and center-crop classification images


In [ ]:
import glob
import os
import SimpleITK as sitk
import numpy as np

DATA_SPLIT = "val"  # options: "train", "val", "test"

SPLIT_CONFIG = {
    "train": r"/path/to/workspace/classification_model/train_data",
    "val": r"/path/to/workspace/classification_model/val_data",
    "test": r"/path/to/workspace/classification_model/test_data",
}

image_dir = SPLIT_CONFIG[DATA_SPLIT]
image_paths = glob.glob(os.path.join(image_dir, "*.nii.gz"))
print(DATA_SPLIT, len(image_paths))


In [ ]:
def resample_image(img, target_spacing):
    original_spacing = img.GetSpacing()
    original_size = img.GetSize()
    new_size = [
        int(np.round(original_size[i] * (original_spacing[i] / target_spacing[i])))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetInterpolator(sitk.sitkLinear)
    return resampler.Execute(img)


In [ ]:
target_spacing = (1.0, 1.0, 1.0)
resampled_images = []

for img_path in image_paths:
    img = sitk.ReadImage(img_path)
    resampled = resample_image(img, target_spacing)
    resampled_images.append({"img_path": img_path, "image": resampled})

print("resampled:", len(resampled_images))
